### 这个notebook只针对 - CustomerReview进行数据预处理

### 连接数据库

In [1]:
import re,emoji
import pandas as pd
from sqlalchemy import create_engine, text
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from langdetect import detect, LangDetectException
from deep_translator import GoogleTranslator

# 数据库配置
username = "postgres"
password = "123456"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)


### 读取数据

In [2]:
# 读取来自CustomerReview的数据
customer_review_sql = """
SELECT * FROM "CustomerReview"
"""

df_cr = pd.read_sql(customer_review_sql, engine)

# 查看数据
df_cr

,review_id,customer_id,product_id,order_id,rating,review_title,review_text,review_date
0,1,68571,54,325330,5,"Strong, soft,and cute. Just what I wanted in a...","I had just gotten my first pair of glasses, an...",2024-05-18
1,2,20501,68,195701,5,Perfect,Don’t have to worry about losing eyeglasses wh...,2024-07-11
2,3,107144,28,463027,5,Great Product For Content Creators,It’s a great product for what it is I just isn...,2025-03-05
3,4,94811,154,17304,5,The best,The Hope they improve quality control.,2024-02-26
4,5,158516,172,236244,5,Great value. Works great!,Nice night driving glasses. Very lightweight. ...,2025-05-14
...,...,...,...,...,...,...,...,...
49995,49996,147565,172,1351,5,Want more,Perfect. Want to order more but they are sold ...,2025-01-07
49996,49997,132810,72,31029,5,Good grow room glasses.,Nice fit with a bit of flexibility in the leng...,2025-06-10
49997,49998,116817,53,92990,5,Five Stars,"fit well, and didn't bounce around while I ran.",2024-06-05
49998,49999,41121,25,15684,5,Worth it!,Love everything this does!!! Only complaint is...,2024-06-11


In [3]:
df_cr.columns.tolist()

['review_id',
 'customer_id',
 'product_id',
 'order_id',
 'rating',
 'review_title',
 'review_text',
 'review_date']

### 文本预处理 - 文本拼接

In [4]:
# ---------- 2) 文本拼接、时间统一、去重
def preprocess_basic_reviews(df):
    # 拼接 title + text，去重，时间解析
    df = df.copy()
    df['review_title'] = df.get('review_title', '').fillna('')
    df['review_text'] = df.get('review_text', '').fillna('')
    df['text'] = (df['review_title'].astype(str) + ' ' + df['review_text'].astype(str)).str.strip()
    if 'review_date' in df.columns:
        df['review_date'] = pd.to_datetime(df['review_date'], errors='coerce')
    # 去重基于 review_id 或 text+customer_id+date
    if 'review_id' in df.columns:
        df = df.drop_duplicates(subset=['review_id'])
    else:
        df = df.drop_duplicates(subset=['customer_id','product_id','text'])
    return df

# ---------- 3) 简单文本去噪函数
def clean_text(s, 
               lower=True, 
               remove_urls=True, 
               remove_html=True, 
               remove_nonprint=True, 
               remove_emoji=True,
               keep_only_english_digits=False):
    if pd.isna(s): return ""
    # text = str(s)
    text = str(s).strip()
    if remove_html:
        text = re.sub(r'<[^>]+>', ' ', text)
    if remove_urls:
        text = re.sub(r'http\S+|www\.\S+', ' ', text)
    
    if remove_emoji and emoji is not None:
        text = emoji.demojize(text, delimiters=(" ", " "))
        text = text.replace("_", " ")
        text = re.sub(r'[:]+', ' ', text)
    if remove_nonprint:
        text = re.sub(r'[\r\n\t]+', ' ', text)
    # 移除常见占位文本 N/A, n.a., NA, na, 等（不区分大小写）
    text = re.sub(r'\b(n/?a|n\.a\.|na)\b', ' ', text, flags=re.I)
    
    text = re.sub(r'\s+', ' ', text).strip()

    if lower:
        text = text.lower()
    if keep_only_english_digits:
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text




### 检测非英文文本内容

### 主程序

In [5]:
# 文本预处理
customer_reviews = preprocess_basic_reviews(df_cr)
# 选择1：清洗文本列，保留原文
customer_reviews['text_clean'] = customer_reviews['text'].apply(clean_text)

# 只检测唯一文本，避免重复计算
unique_texts = customer_reviews["text_clean"].dropna().drop_duplicates()

lang_map = {}
for txt in unique_texts:
    if not isinstance(txt, str):
        lang_map[txt] = "unknown"
        continue
    txt = txt.strip()
    if len(txt) < 2:
        lang_map[txt] = "unknown"
        continue
    try:
        lang_map[txt] = detect(txt)
    except Exception:
        lang_map[txt] = "unknown"

# 回填到原表
customer_reviews["lang"] = customer_reviews["text_clean"].map(lang_map)


In [6]:
customer_reviews

,review_id,customer_id,product_id,order_id,rating,review_title,review_text,review_date,text,text_clean,lang
0,1,68571,54,325330,5,"Strong, soft,and cute. Just what I wanted in a...","I had just gotten my first pair of glasses, an...",2024-05-18,"Strong, soft,and cute. Just what I wanted in a...","strong, soft,and cute. just what i wanted in a...",en
1,2,20501,68,195701,5,Perfect,Don’t have to worry about losing eyeglasses wh...,2024-07-11,Perfect Don’t have to worry about losing eyegl...,perfect don’t have to worry about losing eyegl...,en
2,3,107144,28,463027,5,Great Product For Content Creators,It’s a great product for what it is I just isn...,2025-03-05,Great Product For Content Creators It’s a grea...,great product for content creators it’s a grea...,en
3,4,94811,154,17304,5,The best,The Hope they improve quality control.,2024-02-26,The best The Hope they improve quality control.,the best the hope they improve quality control.,en
4,5,158516,172,236244,5,Great value. Works great!,Nice night driving glasses. Very lightweight. ...,2025-05-14,Great value. Works great! Nice night driving g...,great value. works great! nice night driving g...,en
...,...,...,...,...,...,...,...,...,...,...,...
49995,49996,147565,172,1351,5,Want more,Perfect. Want to order more but they are sold ...,2025-01-07,Want more Perfect. Want to order more but they...,want more perfect. want to order more but they...,en
49996,49997,132810,72,31029,5,Good grow room glasses.,Nice fit with a bit of flexibility in the leng...,2025-06-10,Good grow room glasses. Nice fit with a bit of...,good grow room glasses. nice fit with a bit of...,en
49997,49998,116817,53,92990,5,Five Stars,"fit well, and didn't bounce around while I ran.",2024-06-05,"Five Stars fit well, and didn't bounce around ...","five stars fit well, and didn't bounce around ...",en
49998,49999,41121,25,15684,5,Worth it!,Love everything this does!!! Only complaint is...,2024-06-11,Worth it! Love everything this does!!! Only co...,worth it! love everything this does!!! only co...,en


### 到了这步，属于文本预处理完成了。

In [ ]:
import pandas as pd

df_trans = pd.read_csv("customer_reviews_non_english_translated.csv", encoding="utf-8-sig")
df_trans.columns = df_trans.columns.str.strip()
assert 'text_clean' in df_trans.columns and 'text_clean_translated' in df_trans.columns

mapping = dict(zip(df_trans['text_clean'].astype(str).str.strip(),
                   df_trans['text_clean_translated'].astype(str)))

keys = customer_reviews['text_clean'].astype(str).str.strip()
mapped = keys.map(mapping)           # 未匹配的为 NaN

customer_reviews['text_clean_translated'] = mapped

# 未匹配的（NaN）用原始清洗文本回填
customer_reviews['text_clean_translated'] = customer_reviews['text_clean_translated'].fillna(customer_reviews['text_clean'].astype(str))
customer_reviews_trans = customer_reviews.copy()

# 可选保存
customer_reviews_trans.to_csv("customer_reviews_with_translations.csv", index=False, encoding="utf-8-sig")

In [8]:
customer_reviews_trans

,review_id,customer_id,product_id,order_id,rating,review_title,review_text,review_date,text,text_clean,lang,text_clean_translated
0,1,68571,54,325330,5,"Strong, soft,and cute. Just what I wanted in a...","I had just gotten my first pair of glasses, an...",2024-05-18,"Strong, soft,and cute. Just what I wanted in a...","strong, soft,and cute. just what i wanted in a...",en,"strong, soft,and cute. just what i wanted in a..."
1,2,20501,68,195701,5,Perfect,Don’t have to worry about losing eyeglasses wh...,2024-07-11,Perfect Don’t have to worry about losing eyegl...,perfect don’t have to worry about losing eyegl...,en,perfect don’t have to worry about losing eyegl...
2,3,107144,28,463027,5,Great Product For Content Creators,It’s a great product for what it is I just isn...,2025-03-05,Great Product For Content Creators It’s a grea...,great product for content creators it’s a grea...,en,great product for content creators it’s a grea...
3,4,94811,154,17304,5,The best,The Hope they improve quality control.,2024-02-26,The best The Hope they improve quality control.,the best the hope they improve quality control.,en,the best the hope they improve quality control.
4,5,158516,172,236244,5,Great value. Works great!,Nice night driving glasses. Very lightweight. ...,2025-05-14,Great value. Works great! Nice night driving g...,great value. works great! nice night driving g...,en,great value. works great! nice night driving g...
...,...,...,...,...,...,...,...,...,...,...,...,...
49995,49996,147565,172,1351,5,Want more,Perfect. Want to order more but they are sold ...,2025-01-07,Want more Perfect. Want to order more but they...,want more perfect. want to order more but they...,en,want more perfect. want to order more but they...
49996,49997,132810,72,31029,5,Good grow room glasses.,Nice fit with a bit of flexibility in the leng...,2025-06-10,Good grow room glasses. Nice fit with a bit of...,good grow room glasses. nice fit with a bit of...,en,good grow room glasses. nice fit with a bit of...
49997,49998,116817,53,92990,5,Five Stars,"fit well, and didn't bounce around while I ran.",2024-06-05,"Five Stars fit well, and didn't bounce around ...","five stars fit well, and didn't bounce around ...",en,"five stars fit well, and didn't bounce around ..."
49998,49999,41121,25,15684,5,Worth it!,Love everything this does!!! Only complaint is...,2024-06-11,Worth it! Love everything this does!!! Only co...,worth it! love everything this does!!! only co...,en,worth it! love everything this does!!! only co...


In [9]:
rating_dist = customer_reviews_trans["rating"].value_counts().sort_index()
print("rating 分布：")
print(rating_dist)
print("\n占比：")
print((customer_reviews_trans["rating"].value_counts(normalize=True).sort_index() * 100).round(2))

rating 分布：
rating
1     5294
2     2929
3     4834
4     8223
5    28720
Name: count, dtype: int64

占比：
rating
1    10.59
2     5.86
3     9.67
4    16.45
5    57.44
Name: proportion, dtype: float64


### 数据标签构造

In [10]:
# ---------- 4) 标签构造（示例映射）
def map_review_label_from_rating(df, pos_thresh=4, neg_thresh=2, keep_neutral=False):
    df = df.copy()
    df = df[df['rating'].notna()]
    # 映射：1 positive, 0 negative, 2 neutral (可选)
    def map_func(r):
        try:
            r2 = float(r)
        except (ValueError, TypeError):
            return None
        if r2 >= pos_thresh:
            return 1        # positive
        elif r2 <= neg_thresh:
            return 0        # negative
        else:
            return 2        # neutral

    # 中间状态
    df['label_raw'] = df['rating'].apply(map_func)

    if not keep_neutral:
        # 只保留正负样本（二分类）
        df = df[df['label_raw'].isin([0,1])]

    # 最终标签
    df['label'] = df['label_raw'].astype(int)
    
    # 打印分布，方便检查
    print("标签分布：")
    print(df['label'].value_counts().sort_index())
    print(f"总样本数：{len(df)}")

    return df

In [11]:
# 构造标签：评论的例子
customer_reviews_trans_labeled = map_review_label_from_rating(customer_reviews_trans, pos_thresh=5, neg_thresh=2, keep_neutral=False)
customer_reviews_trans_labeled

标签分布：
label
0     8223
1    28720
Name: count, dtype: int64
总样本数：36943


,review_id,customer_id,product_id,order_id,rating,review_title,review_text,review_date,text,text_clean,lang,text_clean_translated,label_raw,label
0,1,68571,54,325330,5,"Strong, soft,and cute. Just what I wanted in a...","I had just gotten my first pair of glasses, an...",2024-05-18,"Strong, soft,and cute. Just what I wanted in a...","strong, soft,and cute. just what i wanted in a...",en,"strong, soft,and cute. just what i wanted in a...",1,1
1,2,20501,68,195701,5,Perfect,Don’t have to worry about losing eyeglasses wh...,2024-07-11,Perfect Don’t have to worry about losing eyegl...,perfect don’t have to worry about losing eyegl...,en,perfect don’t have to worry about losing eyegl...,1,1
2,3,107144,28,463027,5,Great Product For Content Creators,It’s a great product for what it is I just isn...,2025-03-05,Great Product For Content Creators It’s a grea...,great product for content creators it’s a grea...,en,great product for content creators it’s a grea...,1,1
3,4,94811,154,17304,5,The best,The Hope they improve quality control.,2024-02-26,The best The Hope they improve quality control.,the best the hope they improve quality control.,en,the best the hope they improve quality control.,1,1
4,5,158516,172,236244,5,Great value. Works great!,Nice night driving glasses. Very lightweight. ...,2025-05-14,Great value. Works great! Nice night driving g...,great value. works great! nice night driving g...,en,great value. works great! nice night driving g...,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,49996,147565,172,1351,5,Want more,Perfect. Want to order more but they are sold ...,2025-01-07,Want more Perfect. Want to order more but they...,want more perfect. want to order more but they...,en,want more perfect. want to order more but they...,1,1
49996,49997,132810,72,31029,5,Good grow room glasses.,Nice fit with a bit of flexibility in the leng...,2025-06-10,Good grow room glasses. Nice fit with a bit of...,good grow room glasses. nice fit with a bit of...,en,good grow room glasses. nice fit with a bit of...,1,1
49997,49998,116817,53,92990,5,Five Stars,"fit well, and didn't bounce around while I ran.",2024-06-05,"Five Stars fit well, and didn't bounce around ...","five stars fit well, and didn't bounce around ...",en,"five stars fit well, and didn't bounce around ...",1,1
49998,49999,41121,25,15684,5,Worth it!,Love everything this does!!! Only complaint is...,2024-06-11,Worth it! Love everything this does!!! Only co...,worth it! love everything this does!!! only co...,en,worth it! love everything this does!!! only co...,1,1


### 划分训练集和测试集

In [12]:
from sklearn.model_selection import train_test_split

X = customer_reviews_trans_labeled['text_clean_translated']
y = customer_reviews_trans_labeled['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # 重要：保持正负比例一致
)

print("训练集标签分布：")
print(y_train.value_counts())
print("\n测试集标签分布：")
print(y_test.value_counts())

训练集标签分布：
label
1    22976
0     6578
Name: count, dtype: int64

测试集标签分布：
label
1    5744
0    1645
Name: count, dtype: int64


### TF-IDF + Logistic Regression 训练

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# 构建 Pipeline
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20000,      
        ngram_range=(1, 2),      
        min_df=3,                
        max_df=0.9
    )),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42,
    ))
])

# 训练
pipe.fit(X_train, y_train)

# 预测
y_pred = pipe.predict(X_test)

# 模型评估
print("TF-IDF + Logistic Regression 的分类报告：")
print(classification_report(y_test, y_pred, target_names=['Negative (0)', 'Positive (1)']))

print("\n混淆矩阵：")
print(confusion_matrix(y_test, y_pred))

print("\nMacro F1:", f1_score(y_test, y_pred, average='macro'))
print("Negative F1:", f1_score(y_test, y_pred, pos_label=0))
print("Positive F1:", f1_score(y_test, y_pred, pos_label=1))

TF-IDF + Logistic Regression 的分类报告：
              precision    recall  f1-score   support

Negative (0)       0.89      0.97      0.93      1645
Positive (1)       0.99      0.97      0.98      5744

    accuracy                           0.97      7389
   macro avg       0.94      0.97      0.95      7389
weighted avg       0.97      0.97      0.97      7389


混淆矩阵：
[[1601   44]
 [ 194 5550]]

Macro F1: 0.9549112984940538
Negative F1: 0.9308139534883721
Positive F1: 0.9790086434997354


### 跑 VADER 做无监督对比

In [14]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def vader_label(text):
    score = analyzer.polarity_scores(text)['compound']
    return 1 if score >= 0.05 else 0 if score <= -0.05 else -1  # -1 表示中性

# 只在测试集上对比
vader_pred = X_test.apply(vader_label)

# 过滤掉 VADER 判断为中性的样本后比较
mask = vader_pred != -1
print("VADER 与严格标签的对比（仅非中性样本）：")
print(classification_report(y_test[mask], vader_pred[mask], target_names=['Negative', 'Positive']))

print("\nMacro F1:", f1_score(y_test[mask], vader_pred[mask], average='macro'))
print("Negative F1:", f1_score(y_test[mask], vader_pred[mask], pos_label=0))
print("Positive F1:", f1_score(y_test[mask], vader_pred[mask], pos_label=1))


VADER 与严格标签的对比（仅非中性样本）：
              precision    recall  f1-score   support

    Negative       0.89      0.54      0.68      1452
    Positive       0.89      0.98      0.94      5595

    accuracy                           0.89      7047
   macro avg       0.89      0.76      0.81      7047
weighted avg       0.89      0.89      0.88      7047


Macro F1: 0.8054105949369811
Negative F1: 0.6754910333048676
Positive F1: 0.9353301565690946


### Textblob (NaiveBayesAnalyzer)

In [15]:
from textblob import TextBlob
from textblob.sentiments import NaiveBayesAnalyzer
from sklearn.metrics import classification_report

nb_analyzer = NaiveBayesAnalyzer()

def textblob_nb_label(text):
    try:
        blob = TextBlob(str(text), analyzer=nb_analyzer)
        cls = blob.sentiment.classification  # 'pos' or 'neg'
        return 1 if cls == 'pos' else 0
    except Exception:
        return None

# 在测试集上预测
tb_nb_pred = X_test.apply(textblob_nb_label)

# 过滤掉异常返回的 None（若有）
mask_tb_nb = tb_nb_pred.notna()
print("TextBlob NaiveBayesAnalyzer 与严格标签的对比：")
print(classification_report(y_test[mask_tb_nb], tb_nb_pred[mask_tb_nb], target_names=['Negative', 'Positive']))

print("\nMacro F1:", f1_score(y_test[mask_tb_nb], tb_nb_pred[mask_tb_nb], average='macro'))
print("Negative F1:", f1_score(y_test[mask_tb_nb], tb_nb_pred[mask_tb_nb], pos_label=0))
print("Positive F1:", f1_score(y_test[mask_tb_nb], tb_nb_pred[mask_tb_nb], pos_label=1))

TextBlob NaiveBayesAnalyzer 与严格标签的对比：
              precision    recall  f1-score   support

    Negative       0.36      0.49      0.42      1645
    Positive       0.84      0.76      0.80      5744

    accuracy                           0.70      7389
   macro avg       0.60      0.62      0.61      7389
weighted avg       0.73      0.70      0.71      7389


Macro F1: 0.6062040581393955
Negative F1: 0.4172736732570239
Positive F1: 0.795134443021767


### Textblob (默认的PatternAnalyzer)

In [16]:
from textblob import TextBlob
from textblob.sentiments import PatternAnalyzer
from sklearn.metrics import classification_report

pattern_analyzer = PatternAnalyzer()

def textblob_pattern_label(text, thresh=0.0):
    try:
        blob = TextBlob(str(text), analyzer=pattern_analyzer)
        pol = blob.sentiment.polarity  # 连续值，范围约在[-1, 1]
        if pol > thresh:
            return 1
        elif pol < -thresh:
            return 0
        else:
            return None
    except Exception:
        return None

# 在测试集上预测
tb_pa_pred = X_test.apply(textblob_pattern_label)

# 过滤掉 None（若有）
mask_pa_tb = tb_pa_pred.notna()
print("TextBlob PatternAnalyzer 与严格标签的对比：")
print(classification_report(y_test[mask_pa_tb], tb_pa_pred[mask_pa_tb], target_names=['Negative', 'Positive']))

print("\nMacro F1:", f1_score(y_test[mask_pa_tb], tb_pa_pred[mask_pa_tb], average='macro'))
print("Negative F1:", f1_score(y_test[mask_pa_tb], tb_pa_pred[mask_pa_tb], pos_label=0))
print("Positive F1:", f1_score(y_test[mask_pa_tb], tb_pa_pred[mask_pa_tb], pos_label=1))

TextBlob PatternAnalyzer 与严格标签的对比：
              precision    recall  f1-score   support

    Negative       0.89      0.44      0.59      1497
    Positive       0.87      0.99      0.92      5540

    accuracy                           0.87      7037
   macro avg       0.88      0.71      0.75      7037
weighted avg       0.87      0.87      0.85      7037


Macro F1: 0.7538203331778548
Negative F1: 0.5857590685176892
Positive F1: 0.9218815978380205


### 模型保存

In [ ]:
# 保存模型（可选）
import joblib
joblib.dump(pipe, "tfidf_logreg_strict.pkl")

In [15]:
import sys

print(sys.executable)
print(sys.version)

d:\miniconda_envs\junliangvenv_nlp1\python.exe
3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]


In [16]:
import importlib.util

spec = importlib.util.find_spec("torch")

print(spec.origin)

d:\miniconda_envs\junliangvenv_nlp1\Lib\site-packages\torch\__init__.py


### 测试GPU的连接性

In [17]:
from transformers import pipeline
import torch
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import classification_report, f1_score

device = 0 if torch.cuda.is_available() else -1
print("使用设备：", "GPU" if device == 0 else "CPU")

d:\miniconda_envs\junliangvenv_nlp1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


使用设备： GPU


In [18]:
import torch, sys
print("python:", sys.executable)
print("torch:", torch.__version__)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.version.cuda:", torch.version.cuda)
print("torch.backends.cudnn.version():", torch.backends.cudnn.version())

python: d:\miniconda_envs\junliangvenv_nlp1\python.exe
torch: 2.5.1+cu121
torch.cuda.is_available(): True
torch.version.cuda: 12.1
torch.backends.cudnn.version(): 90100


### BERT模型 A：cardiffnlp/twitter-roberta-base-sentiment-latest（英文强）

In [ ]:
cardiffnlp_local_model_path = r"D:\huggingface_models\cardiffnlp-twitter-roberta-safe"

pipe_roberta = pipeline(
    "sentiment-analysis",
    model=cardiffnlp_local_model_path,
    tokenizer=cardiffnlp_local_model_path,
    device=device,
    truncation=True,
    max_length=512,
    batch_size=8, 
)

def predict_roberta(texts):
    results = pipe_roberta(texts)
    # 标签映射：negative → 0, neutral → 忽略或特殊处理, positive → 1
    labels = []
    for r in results:
        if r['label'] == 'negative':
            labels.append(0)
        elif r['label'] == 'positive':
            labels.append(1)
        else:
            labels.append(-1)  # neutral
    return labels

print(pipe_roberta("This product is amazing!"))

[{'label': 'positive', 'score': 0.9836453199386597}]


### BERT模型 B：nlptown/bert-base-multilingual-uncased-sentiment（支持英/法/西）

In [25]:
nlptown_bert_local_model_path = r"D:\huggingface_models\bert-base-multilingual-uncased-sentiment"

pipe_multi = pipeline(
    "sentiment-analysis",
    model=nlptown_bert_local_model_path,
    tokenizer=nlptown_bert_local_model_path,
    device=device,
    truncation=True,
    max_length=512,
    batch_size=8,
)

def predict_multilingual(texts):
    results = pipe_multi(texts)
    labels = []
    for r in results:
        # 输出是 '1 star' ~ '5 star'
        star = int(r['label'].split()[0])
        if star >= 4:
            labels.append(1)      # 正面
        elif star <= 2:
            labels.append(0)      # 负面
        else:
            labels.append(-1)     # 中性（3 star）
    return labels

### BERT模型 C：distilbert-base-uncased-finetuned-sst-2-english（最轻量）

In [ ]:
distilbert_local_model_path = r"D:\huggingface_models\distilbert-base-uncased-finetuned-sst-2-english"

pipe_distil = pipeline(
    "sentiment-analysis",
    model=distilbert_local_model_path,
    tokenizer=distilbert_local_model_path,
    device=device,
    truncation=True,
    max_length=512,
    batch_size=16,
)

def predict_distil(texts):
    results = pipe_distil(texts)
    # 标签：NEGATIVE → 0, POSITIVE → 1
    return [0 if r['label'] == 'NEGATIVE' else 1 for r in results]

### BERT模型评估

In [22]:
def evaluate_model(predict_func, X_test, y_test, model_name="Model"):
    batch_size = 32
    all_preds = []
    
    for i in tqdm(range(0, len(X_test), batch_size), desc=model_name):
        batch = X_test.iloc[i:i+batch_size].tolist()
        preds = predict_func(batch)
        all_preds.extend(preds)
    
    preds = pd.Series(all_preds, index=X_test.index)
    
    # 过滤中性（-1）
    mask = preds != -1
    print(f"\n===== {model_name} 评估结果 =====")
    print(classification_report(y_test[mask], preds[mask], 
                                target_names=['Negative', 'Positive']))
    print("Macro F1:", f1_score(y_test[mask], preds[mask], average='macro'))
    print("Negative F1:", f1_score(y_test[mask], preds[mask], pos_label=0))
    
    return preds

In [23]:
# 英文强模型
preds_roberta = evaluate_model(predict_roberta, X_test, y_test, "Twitter RoBERTa")

Twitter RoBERTa: 100%|██████████| 231/231 [00:51<00:00,  4.47it/s]


===== Twitter RoBERTa 评估结果 =====
              precision    recall  f1-score   support

    Negative       0.96      0.93      0.94      1495
    Positive       0.98      0.99      0.98      5556

    accuracy                           0.98      7051
   macro avg       0.97      0.96      0.96      7051
weighted avg       0.98      0.98      0.98      7051

Macro F1: 0.9634549957451002
Negative F1: 0.94213750850919


In [24]:
# 以多语言模型为例
preds_multi = evaluate_model(predict_multilingual, X_test, y_test, "Multilingual BERT")

Multilingual BERT: 100%|██████████| 231/231 [00:56<00:00,  4.09it/s]


===== Multilingual BERT 评估结果 =====
              precision    recall  f1-score   support

    Negative       0.97      0.98      0.98      1522
    Positive       1.00      0.99      0.99      5666

    accuracy                           0.99      7188
   macro avg       0.98      0.99      0.98      7188
weighted avg       0.99      0.99      0.99      7188

Macro F1: 0.9848716017548139
Negative F1: 0.9761982393218128


In [28]:
# 轻量模型
preds_distil = evaluate_model(predict_distil, X_test, y_test, "DistilBERT")

DistilBERT: 100%|██████████| 231/231 [00:35<00:00,  6.51it/s]


===== DistilBERT 评估结果 =====
              precision    recall  f1-score   support

    Negative       0.82      0.90      0.86      1645
    Positive       0.97      0.94      0.96      5744

    accuracy                           0.93      7389
   macro avg       0.89      0.92      0.91      7389
weighted avg       0.94      0.93      0.93      7389

Macro F1: 0.9059393576460948
Negative F1: 0.855732946298984


### 提取关键词(针对`y_pred`)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text
from collections import Counter
from pathlib import Path
import json

# 假设你已经有：
# X_test: 测试文本
# y_pred: 预测标签
# pipe: 训练好的 Pipeline

# 过滤常见停用词
stop_words = text.ENGLISH_STOP_WORDS

vectorizer = pipe.named_steps['tfidf']
clf = pipe.named_steps['clf']

# 1) 构造评估表
df_eval_y_pred = pd.DataFrame({
    "text": X_test.tolist(),
    "pred_label": y_pred
})

# 2) 按类别分组
df_pos = df_eval_y_pred[df_eval_y_pred["pred_label"] == 1]
df_neg = df_eval_y_pred[df_eval_y_pred["pred_label"] == 0]

# 3) 计算每类文本的 TF-IDF 词频
def top_tfidf_keywords(df, top_n, stop_words=None):
    texts = df["text"].fillna("").tolist()
    if len(texts) == 0:
        return []

    X = vectorizer.transform(texts)
    scores = X.sum(axis=0).A1
    feature_names = vectorizer.get_feature_names_out()

    ranked = []
    for i in range(len(feature_names)):
        term = feature_names[i]
        score = float(scores[i])
        if stop_words is not None:
            t = term.lower()
            if t in stop_words:
                continue
        ranked.append((term, score))

    ranked = sorted(ranked, key=lambda x: x[1], reverse=True)
    return ranked[:top_n]


def save_keyword_counter_outputs(pos_keywords, neg_keywords, output_csv, output_jsonl, model_name):
    rows = []
    for label, kw_list in [("positive", pos_keywords), ("negative", neg_keywords)]:
        for term, score in kw_list:
            rows.append({
                "label": label,
                "keyword": term,
                "score": float(score),
                "source_model": model_name
            })

    df = pd.DataFrame(rows)

    # CSV 输出：适合 Power BI / Excel
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"CSV 已保存到: {Path(output_csv).resolve()}")

    # JSONL 输出：适合 RAG / LLM 知识库
    jsonl_path = Path(output_jsonl)
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            record = {
                "label": row["label"],
                "keyword": row["keyword"],
                "score": float(row["score"]),
                "source_model": row["source_model"],
                "text": (
                    f"{row['label']} keyword: {row['keyword']}; "
                    f"score {float(row['score']):.6f} in {row['source_model']}."
                )
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"JSONL 已保存到: {jsonl_path.resolve()}")
    return df


pos_keywords = top_tfidf_keywords(df_pos, top_n=100, stop_words=stop_words)
neg_keywords = top_tfidf_keywords(df_neg, top_n=100, stop_words=stop_words)

save_keyword_counter_outputs(
    pos_keywords=pos_keywords,
    neg_keywords=neg_keywords,
    output_csv="6-4-customer_review_keywords-by-TF-IDF.csv",
    output_jsonl="6-4-customer_review_keywords-by-TF-IDF.jsonl",
    model_name="TF-IDF+LogReg"
)

print("Positive keywords:")
print(pos_keywords[:20])

print("Negative keywords:")
print(neg_keywords[:20])

CSV 已保存到: G:\圣戈班工作内容\2026-04-15-个人项目的开发思路\2026-04-20-eyewear-data-analysis\6-NLP\6-4-customer_review_keywords-by-TF-IDF.csv
JSONL 已保存到: G:\圣戈班工作内容\2026-04-15-个人项目的开发思路\2026-04-20-eyewear-data-analysis\6-NLP\6-4-customer_review_keywords-by-TF-IDF.jsonl
Positive keywords:
[('great', 195.33038343797594), ('five stars', 168.75406968712116), ('stars', 162.44610035890773), ('love', 146.8620545916477), ('glasses', 140.65029708089656), ('quality', 121.97710618843978), ('good', 115.39898840493758), ('perfect', 88.21661926590518), ('nice', 85.32369092527239), ('fit', 82.20402765637597), ('buy', 76.34112508410945), ('price', 67.14265787805418), ('product', 65.93104072901657), ('service', 65.13415005047844), ('secure', 63.75896297704728), ('cute', 63.56936077508404), ('packaging', 63.04777854730472), ('friendly', 62.76636627463042), ('was friendly', 62.74307826313232), ('excellent', 62.44548741870693)]
Negative keywords:
[('glasses', 37.222892664809855), ('cheap', 33.765411834182004), ('star', 32.

In [31]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("当前设备:", device)

当前设备: cuda


### 提取关键词(针对`preds_roberta`) -v2

In [ ]:
from keybert import KeyBERT
from collections import Counter
from sentence_transformers import SentenceTransformer
from pathlib import Path
import json
import re

model = SentenceTransformer(
    r"D:\huggingface_models\sentence-transformers-all-MiniLM-L6-v2",
    device="cuda"
)

kw_model = KeyBERT(model=model)

PHRASE_BLACKLIST = {
    "improve quality control", "packaging secure", "service friendly",
    "buy brand", "hope improve", "hope improve quality", "quality control",
    "improve quality", "stars love", "stars great", "stars good", "stars nice",
    "make sure", "exactly like", "love love"
}


SENTIMENT_STOPWORDS = {
    "great","love","good","perfect","nice","stars","awesome","excellent",
    "happy","like","really","just","super","best","better","pretty",
    "buy","bought","got","product","use","look","looking","way","time",
    "hope","improve","definitely","absolutely","highly","exactly",
    "star","stars","don","didn","does","did","doesn","is","are",
    "very","so","too","also","even","still","much","many","lot"
}

def is_valid_keyword(term: str, score: float = 1.0) -> bool:
    term = term.lower().strip()
    if not term:
        return False
    if term.isdigit():
        return False
    if term in PHRASE_BLACKLIST:
        return False
    if len(term) < 5:
        return False
    if score < 0.25:
        return False
    bad_patterns = ["don ", "didn ", "does ", " use ", " make ", " want "]
    if any(p in term for p in bad_patterns):
        return False
    return True



# 用英文文本，最好是 text_clean_translated
df_eval_pred_roberta = pd.DataFrame({
    "text": X_test.tolist(),
    "pred_label": [1 if p == 1 else 0 if p == 0 else -1 for p in preds_roberta]
})

# 只保留正负样本
df_eval_pred_roberta = df_eval_pred_roberta[df_eval_pred_roberta["pred_label"] != -1].copy()

# 关键词提取（按类别）
keyword_counter = {
    "positive": Counter(),
    "negative": Counter()
}

for label_name, label_value in {"positive": 1, "negative": 0}.items():
    docs = df_eval_pred_roberta.loc[df_eval_pred_roberta["pred_label"] == label_value, "text"].tolist()

    # 先对每条文本提 top_n 个关键词
    all_terms = []
    for d in docs:
        try:
            terms = kw_model.extract_keywords(
                d,
                keyphrase_ngram_range=(2, 3),
                stop_words='english',
                top_n=8,
                use_mmr=True,
                diversity=0.6
            )
            valid_terms = [term for term, score in terms if is_valid_keyword(term, score)]
            all_terms.extend(valid_terms)
        except Exception:
            pass
    keyword_counter[label_name].update(all_terms)


def save_keyword_counter_outputs(counter_dict, output_csv, output_jsonl, top_n, model_name):
    """
    将关键词计数结果同时导出为：
    1) CSV：适合 Power BI / Excel / 报表分析
    2) JSONL：适合 RAG / LLM 知识库
    """
    rows = []

    for label in ["positive", "negative"]:
        for term, count in counter_dict[label].most_common(top_n):
            rows.append({
                "label": label,
                "keyword": term,
                "count": count,
                "source_model": model_name
            })

    df = pd.DataFrame(rows)

    # 1) CSV 输出：适合 Power BI
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"CSV 已保存到: {Path(output_csv).resolve()}")

    # 2) JSONL 输出：适合 RAG / LLM 知识库
    jsonl_path = Path(output_jsonl)
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            record = {
                "label": row["label"],
                "keyword": row["keyword"],
                "count": int(row["count"]),
                "source_model": row["source_model"],
                "text": (
                    f"{row['label']} keyword: {row['keyword']}; "
                    f"appears {int(row['count'])} times in {row['source_model']} predictions."
                )
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"JSONL 已保存到: {jsonl_path.resolve()}")
    return df

save_keyword_counter_outputs(
    counter_dict=keyword_counter,
    output_csv="6-4-customer_review_keywords_by_Twitter_RoBERTa_v21.csv",
    output_jsonl="6-4-customer_review_keywords_by_Twitter_RoBERTa_v21.jsonl",
    top_n=100,
    model_name="Twitter_RoBERTa"
)
print("Positive keywords:")
print(keyword_counter["positive"].most_common(20))

print("Negative keywords:")
print(keyword_counter["negative"].most_common(20))

CSV 已保存到: G:\圣戈班工作内容\2026-04-15-个人项目的开发思路\2026-04-20-eyewear-data-analysis\6-NLP\6-4-customer_review_keywords_by_Twitter_RoBERTa_v21.csv
JSONL 已保存到: G:\圣戈班工作内容\2026-04-15-个人项目的开发思路\2026-04-20-eyewear-data-analysis\6-NLP\6-4-customer_review_keywords_by_Twitter_RoBERTa_v21.jsonl
Positive keywords:
[('great quality', 89), ('good quality', 86), ('love glasses', 76), ('fit perfect', 60), ('fast shipping', 60), ('vision sharp fit', 58), ('great product', 56), ('perfect fit', 54), ('quality vision sharp', 54), ('shipping glasses look', 51), ('photos great value', 50), ('transition lenses wants', 49), ('camera quality', 49), ('super cute', 49), ('lenses clear frame', 48), ('great glasses', 47), ('good product', 46), ('quality received', 46), ('great price', 46), ('luv fast shipping', 44)]
Negative keywords:
[('poor quality', 49), ('app glasses', 32), ('unbelievable software ready', 32), ('update device useless', 32), ('connect ai function', 32), ('trouble fit comfortable', 26), ('lenses came',

### 提取关键词(针对`preds_roberta`) -v4

In [ ]:
from keybert import KeyBERT
from collections import Counter
from sentence_transformers import SentenceTransformer
from pathlib import Path
import json
import re

model = SentenceTransformer(
    r"D:\huggingface_models\sentence-transformers-all-MiniLM-L6-v2",
    device="cuda"
)

kw_model = KeyBERT(model=model)

PHRASE_BLACKLIST = {
    "improve quality control", "packaging secure", "service friendly",
    "buy brand", "hope improve", "hope improve quality", "quality control",
    "improve quality", "stars love", "stars great", "stars good", "stars nice",
    "make sure", "exactly like", "love love"
}


SENTIMENT_STOPWORDS = {
    "great","love","good","perfect","nice","stars","awesome","excellent",
    "happy","like","really","just","super","best","better","pretty",
    "buy","bought","got","product","use","look","looking","way","time",
    "hope","improve","definitely","absolutely","highly","exactly",
    "star","stars","don","didn","does","did","doesn","is","are",
    "very","so","too","also","even","still","much","many","lot"
}

# 同义/变体映射（左侧为匹配模式，右侧为归一化后词）
dimension_map = {
    # ======= quality 系列（拆成多个维度） =======
    # 总体质量（正）
    r'\b(great|good|high|excellent)\s+quality\b': 'quality_general',
    # 总体质量（负）
    r'\b(poor|bad|terrible|awful)\s+quality\b': 'quality_general_neg',

    # 做工 / 制作质量
    r'\b(build|construction|workmanship)\s+quality\b': 'build_quality',
    r'\bexcellent\s+build\b': 'build_quality',
    r'\bpoor\s+build\b': 'build_quality_neg',
    r'\bcheaply\s+made\b': 'build_quality_neg',

    # 耐用性 / 牢固性
    r'\b(durab(le|ility)|long[-\s]?lasting)\b': 'durability',
    r'\b(excellent|good)\s+durab(le|ility)\b': 'durability',
    r'\b(poor|bad)\s+durab(le|ility)\b': 'durability_neg',
    r'\b(broke|broken|snapped|fell apart)\b': 'durability_neg',

    # 视觉质量 / 视力相关
    r'\b(vision|visual)\s+quality\b': 'vision_quality',
    r'\b(vision|sight)\s+(is\s+)?(clear|sharp)\b': 'vision_quality',
    r'\bclear\s+vision\b': 'vision_quality',
    r'\b(blurry|blurred|distorted)\b': 'vision_quality_neg',

    # 镜片质量（lens）
    r'\b(lens|lenses|coating)\s+quality\b': 'lens_quality',
    r'\b(polarized|anti[-\s]?glare|scratch[-\s]?resistant)\b': 'lens_feature_positive',
    r'\b(scratched|scratches|peel(ing)? off)\b': 'lens_quality_neg',

    # ======= love / like 系列 =======
    r'\b(love|loved|loves)\s+(these\s+)?(glasses|product|them)\b': 'love_product',
    r'\b(handy\s+love|really\s+love|absolutely\s+love)\b': 'love_product',
    r'\b(like|liked|enjoy)\s+(the\s+)?(glasses|product)\b': 'like_product',

    # ======= great 系列（按对象细分） =======
    r'\bgreat\s+(product|glasses|item|thing|order|buy)\b': 'great_product',
    r'\bgreat\s+(price|value|deal)\b': 'great_value',
    r'\bgreat\s+fit\b': 'great_fit',
    r'\bgreat\s+quality\b': 'quality_general',
    r'\bgreat\s+service\b': 'great_service',

    # ======= comfortable 系列 =======
    r'\bcomfortable\s+(to\s+wear|fit|lightweight|for\s+all\s+day)\b': 'comfortable',
    r'\b(very\s+comfortable|so\s+comfortable)\b': 'comfortable',
    r'\bsturdy\s+and\s+comfortable\b': 'comfortable',
    r'\buncomfortable\b': 'comfortable_neg',

    # ======= shipping / delivery 系列 =======
    r'\b(fast|quick|quickly|prompt)\s+(shipping|delivery)\b': 'shipping_fast',
    r'\b(slow|delayed|late)\s+(shipping|delivery)\b': 'shipping_slow',
    r'\b(pack|package)\s+(arrived|came|damaged)\b': 'shipping_condition',

    # ======= prescription / fitting 系列 =======
    r'\b(prescription|rx)\s+(glasses|lenses)\b': 'prescription',
    r'\b(correct|wrong)\s+prescription\b': 'prescription_quality',
    r'\b(need(ed)?\s+prescription|prescription\s+fit)\b': 'prescription_request',

    # ======= price / value 系列 =======
    r'\bcheap\b': 'price_cheap',
    r'\b(overpriced|expensive)\b': 'price_expensive',
    r'\bgood\s+value\b': 'price_value',

    # ======= fit / size 系列 =======
    r'\b(too\s+small|too\s+big|too\s+large|too\s+tight|too\s+loose)\b': 'fit_issue',
    r'\b(perfect\s+fit|fits\s+perfectly|fits\s+well)\b': 'fit_ok',

    # ======= general sentiment words that often need context =======
    r'\bgreat\b': 'sent_great',   # 可作为弱信号，需与对象联合判断
    r'\bgood\b': 'sent_good',
    r'\bbad\b': 'sent_bad',
    r'\bexcellent\b': 'sent_excellent',
    r'\bterrible\b': 'sent_terrible',

    # ======= 其它常见短语 / 防噪声（例：无关或模板句） =======
    r'\b(return(ed)?|refund|warranty)\b': 'after_sale',
    r'\b(customer\s+service|support)\b': 'service',
}

def normalize_term(term: str) -> str:
    t = term.lower().strip()
    # 先简单清理多余空格和标点
    t = re.sub(r"[^\w\s']", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    # 先尝试完全匹配映射（提高精确度）
    for patt, norm in dimension_map.items():
        if re.search(patt, t):
            return norm
    return t


def merge_counter(counter: Counter) -> Counter:
    new_c = Counter()
    for term, cnt in counter.items():
        norm = normalize_term(term)
        if norm:
            new_c[norm] += cnt
    return new_c

def is_valid_keyword(term: str, score: float = 1.0) -> bool:
    term = term.lower().strip()
    tokens = term.split()
    if not term:
        return False
    bad_edges = {
        'thing','ones','does','did','doesn','didn','sure',
        'justify','kind','way','really','don','quality','received'
    }
    if tokens[0] in bad_edges or tokens[-1] in bad_edges:
        return False
    if 'quality quality' in term or 'quality does' in term:
        return False

    if term.isdigit():
        return False
    if term in PHRASE_BLACKLIST or term in SENTIMENT_STOPWORDS:
        return False
    if len(term) < 4 and term not in dimension_map:
        return False
    if score < 0.25:
        return False
    bad_patterns = ["don", "didn", "does", " use", " make", " want"]
    if any(p in term for p in bad_patterns):
        return False
    return True



# 用英文文本，最好是 text_clean_translated
df_eval_pred_roberta = pd.DataFrame({
    "text": X_test.tolist(),
    "pred_label": [1 if p == 1 else 0 if p == 0 else -1 for p in preds_roberta]
})

# 只保留正负样本
df_eval_pred_roberta = df_eval_pred_roberta[df_eval_pred_roberta["pred_label"] != -1].copy()

# 关键词提取（按类别）
keyword_counter = {
    "positive": Counter(),
    "negative": Counter()
}

for label_name, label_value in {"positive": 1, "negative": 0}.items():
    docs = df_eval_pred_roberta.loc[df_eval_pred_roberta["pred_label"] == label_value, "text"].tolist()

    # 先对每条文本提 top_n 个关键词
    all_terms = []
    for d in docs:
        try:
            terms = kw_model.extract_keywords(
                d,
                keyphrase_ngram_range=(2, 3),
                stop_words='english',
                top_n=8,
                use_mmr=True,
                diversity=0.6
            )
            valid_terms = [term for term, score in terms if is_valid_keyword(term, score)]
            norm_terms = [normalize_term(t) for t in valid_terms]
            norm_terms = [t for t in norm_terms if t]
            all_terms.extend(norm_terms)

        except Exception:
            pass
    keyword_counter[label_name].update(all_terms)




def save_keyword_counter_outputs(counter_dict, output_csv, output_jsonl, top_n, model_name):
    """
    将关键词计数结果同时导出为：
    1) CSV：适合 Power BI / Excel / 报表分析
    2) JSONL：适合 RAG / LLM 知识库
    """
    rows = []

    for label in ["positive", "negative"]:
        for term, count in counter_dict[label].most_common(top_n):
            rows.append({
                "label": label,
                "keyword": term,
                "count": count,
                "source_model": model_name
            })

    df = pd.DataFrame(rows)

    # 1) CSV 输出：适合 Power BI
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"CSV 已保存到: {Path(output_csv).resolve()}")

    # 2) JSONL 输出：适合 RAG / LLM 知识库
    jsonl_path = Path(output_jsonl)
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            record = {
                "label": row["label"],
                "keyword": row["keyword"],
                "count": int(row["count"]),
                "source_model": row["source_model"],
                "text": (
                    f"{row['label']} keyword: {row['keyword']}; "
                    f"appears {int(row['count'])} times in {row['source_model']} predictions."
                )
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"JSONL 已保存到: {jsonl_path.resolve()}")
    return df


save_keyword_counter_outputs(
    counter_dict=keyword_counter,
    output_csv="6-4-customer_review_keywords_by_Twitter_RoBERTa_v25.csv",
    output_jsonl="6-4-customer_review_keywords_by_Twitter_RoBERTa_v25.jsonl",
    top_n=100,
    model_name="Twitter_RoBERTa"
)
print("Positive keywords:")
print(keyword_counter["positive"].most_common(20))

print("Negative keywords:")
print(keyword_counter["negative"].most_common(20))

CSV 已保存到: G:\圣戈班工作内容\2026-04-15-个人项目的开发思路\2026-04-20-eyewear-data-analysis\6-NLP\6-4-customer_review_keywords_by_Twitter_RoBERTa_v25.csv
JSONL 已保存到: G:\圣戈班工作内容\2026-04-15-个人项目的开发思路\2026-04-20-eyewear-data-analysis\6-NLP\6-4-customer_review_keywords_by_Twitter_RoBERTa_v25.jsonl
Positive keywords:
[('sent_great', 1816), ('sent_good', 1150), ('great_product', 446), ('sent_excellent', 435), ('love_product', 389), ('great_value', 273), ('shipping_fast', 197), ('durability', 177), ('quality_general', 176), ('fit_ok', 173), ('lens_feature_positive', 141), ('comfortable', 125), ('vision_quality', 121), ('price_cheap', 101), ('great_fit', 91), ('price_expensive', 80), ('build_quality', 73), ('fit perfect', 60), ('price_value', 52), ('shipping glasses look', 51)]
Negative keywords:
[('durability_neg', 398), ('price_cheap', 284), ('lens_quality_neg', 136), ('sent_good', 106), ('after_sale', 73), ('vision_quality_neg', 70), ('durability', 49), ('comfortable_neg', 48), ('sent_bad', 42), ('lens_feat

### 提取关键词(针对`preds_roberta`) -v3

In [ ]:
from keybert import KeyBERT
from collections import Counter, defaultdict
from sentence_transformers import SentenceTransformer
from pathlib import Path
import json
import re
import pandas as pd
import spacy

# ========== 0. 加载模型 ==========
model = SentenceTransformer(
    r"D:\huggingface_models\sentence-transformers-all-MiniLM-L6-v2",
    device="cuda"
)
kw_model = KeyBERT(model=model)
nlp = spacy.load("en_core_web_sm")

# ========== 1. 构造 df + 清洗 ==========
df_eval_pred_roberta = pd.DataFrame({
    "text": X_test.tolist(),
    "pred_label": [1 if p == 1 else 0 if p == 0 else -1 for p in preds_roberta]
})
df_eval_pred_roberta = df_eval_pred_roberta[
    df_eval_pred_roberta["pred_label"] != -1
].copy()

# 去重 + 去短文本
df_eval_pred_roberta = df_eval_pred_roberta.drop_duplicates(subset=["text"])
df_eval_pred_roberta = df_eval_pred_roberta[
    df_eval_pred_roberta["text"].str.len() > 10
].copy()

# 剔除串味文本（按你实际排查结果调整关键词）
BAD_PATTERN = r"vpn|shoes carpet|ai function|work europe|amazon value|update device|light disabled"
df_eval_pred_roberta = df_eval_pred_roberta[
    ~df_eval_pred_roberta["text"].str.contains(BAD_PATTERN, case=False, na=False)
].copy()

# ========== 2. 黑名单：纯情感词 + 功能词 + 品类词 ==========
STOP_EXTRA = {
    # 纯情感词（积极）
    "great","good","love","loves","loved","nice","perfect","perfectly","happy",
    "awesome","excellent","beautiful","super","cool","pretty","best","better",
    "amazing","wonderful","fantastic","glad","enjoy","enjoyed",
    # 纯情感词（消极）
    "poor","bad","disappointed","disappointing","cheap","junk","waste","terrible",
    "awful","horrible","sad","unhappy","hate","hated","worst","worse",
    # 功能词/副词/动词
    "just","really","very","quite","too","also","even","still","yet","however",
    "got","get","buy","bought","purchase","purchased","use","used","using",
    "work","works","worked","look","looks","looking","way","time","times",
    "hope","hoping","improve","improved","recommend","recommended","received",
    "don","didn","does","did","doesn","isn","wasn","aren","weren",
    # 评分符号
    "star","stars","rating","review","reviews",
    # 品类词（眼镜行业，按你实际情况调整）
    "glasses","glass","sunglasses","reading glasses","product","products",
    "brand","brands","price","prices","pair","pairs","lenses","lens","frame",
    "frames","prescription","case","face","son","husband","wife","daughter",
    "shoes","app","device","video","vpn",
}

# ========== 3. 抽“方面词”：名词短语 + KeyBERT 交集 ==========
def extract_aspects(text):
    """先 KeyBERT 抽候选，再用 spaCy 只保留名词短语"""
    try:
        terms = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(2, 3),
            stop_words='english',
            top_n=8,
            use_mmr=True,
            diversity=0.6
        )
        candidates = [t for t, _ in terms]
    except Exception:
        candidates = []

    doc = nlp(text)
    noun_chunks = {chunk.text.lower().strip() for chunk in doc.noun_chunks}

    aspects = []
    for c in candidates:
        c_low = c.lower().strip()
        # 过滤黑名单
        if c_low in STOP_EXTRA:
            continue
        # 过滤单字/纯数字
        if len(c_low) < 3 or c_low.isdigit():
            continue
        # 优先保留名词短语；如果不是名词短语但也通过黑名单，也保留（宽松模式）
        aspects.append(c_low)
    return aspects

# ========== 4. 统计：每个方面词在正/负里的次数 ==========
aspect_stats = defaultdict(lambda: {"pos": 0, "neg": 0})

for text, label in zip(df_eval_pred_roberta["text"],
                       df_eval_pred_roberta["pred_label"]):
    label_name = "pos" if label == 1 else "neg"
    for asp in set(extract_aspects(text)):   # set 去重，避免同条文本重复计数
        aspect_stats[asp][label_name] += 1

# ========== 5. 生成结果表 ==========
rows = []
for asp, c in aspect_stats.items():
    total = c["pos"] + c["neg"]
    if total < 10:          # 过滤低频方面
        continue
    rows.append({
        "aspect": asp,
        "pos_count": c["pos"],
        "neg_count": c["neg"],
        "total": total,
        "neg_ratio": round(c["neg"] / total, 3),
        "pos_ratio": round(c["pos"] / total, 3),
    })

result_df = pd.DataFrame(rows).sort_values("total", ascending=False)

# 分两张表：最常被夸 / 最常被骂
top_positive = result_df.sort_values(
    ["pos_count", "pos_ratio"], ascending=False
).head(30)

top_negative = result_df.sort_values(
    ["neg_count", "neg_ratio"], ascending=False
).head(30)

print("=== 最常被夸的方面 ===")
print(top_positive)
print("\n=== 最常被骂的方面 ===")
print(top_negative)

# ========== 6. 导出 ==========
result_df.to_csv("6-4-aspect_sentiment_stats.csv", index=False, encoding="utf-8-sig")
top_positive.to_csv("6-4-aspect_top_positive.csv", index=False, encoding="utf-8-sig")
top_negative.to_csv("6-4-aspect_top_negative.csv", index=False, encoding="utf-8-sig")

=== 最常被夸的方面 ===
                      aspect  pos_count  neg_count  total  neg_ratio  \
4    improve quality control        322         90    412      0.218   
2           packaging secure        310         55    365      0.151   
0           service friendly        299         58    357      0.162   
14                 buy brand        288         72    360      0.200   
45              hope improve        153         51    204      0.250   
36      hope improve quality        121         40    161      0.248   
30                stars love        117          0    117      0.000   
22           quality control        107         46    153      0.301   
93              good quality         90          4     94      0.043   
25               stars great         82          0     82      0.000   
29             great quality         77          2     79      0.025   
38           improve quality         69         16     85      0.188   
11              love glasses         64         

### 提取关键词(针对`preds_multi`)

### 1 准备数据

In [42]:
label_map = {1: "positive", 0: "negative", -1: "neutral"}
df_eval_preds_multi = pd.DataFrame({
    "text": X_test.tolist(),
    "true_label": y_test.tolist(),
    "pred_label": [label_map[p] for p in preds_multi]
})
df_eval_preds_multi["pred_label"].value_counts()
df_eval_preds_multi

,text,true_label,pred_label
0,simple the glasses were ok. they didn’t make a...,0,neutral
1,fog up easily. thin plastic but nice and light...,1,neutral
2,less fatigue and blurinezs i am at a computer ...,1,positive
3,computer glasses received very quickly and imm...,1,positive
4,five stars my glasses was the perfect accessor...,1,positive
...,...,...,...
7384,love these glasses very comfortable and lightw...,1,positive
7385,what a find i collect comics so these a a grea...,1,positive
7386,highly recommend! a bit heavy but very fashion...,1,positive
7387,would purchase again love these way more than ...,1,positive


### 2 使用 LIME 解释 `preds_multi`，对前 100 条文本解释

In [ ]:
from lime.lime_text import LimeTextExplainer
import numpy as np
from collections import Counter
label_map = {1: "positive", 0: "negative", -1: "neutral"}
df_eval_preds_multi = pd.DataFrame({
    "text": X_test.tolist(),
    "true_label": y_test.tolist(),
    "pred_label": [label_map[p] for p in preds_multi]
})
df_eval_preds_multi["pred_label"].value_counts()


def predict_proba_multilingual(texts):
    results = pipe_multi(texts, top_k=5)
    probs = []
    for item in results:
        score_map = {r["label"]: r["score"] for r in item}
        neg = score_map.get("1 star", 0.0) + score_map.get("2 star", 0.0)
        neut = score_map.get("3 star", 0.0)
        pos = score_map.get("4 star", 0.0) + score_map.get("5 star", 0.0)
        probs.append([neg, neut, pos])
    return np.array(probs)

explainer = LimeTextExplainer(class_names=["negative", "neutral", "positive"])
explainer


max_samples = min(5, len(df_eval_preds_multi))
class_counters = {
    "negative": Counter(),
    "neutral": Counter(),
    "positive": Counter()
}
rows = []

for i in range(max_samples):
    text = df_eval_preds_multi.loc[i, "text"]
    pred_label = df_eval_preds_multi.loc[i, "pred_label"]
    exp = explainer.explain_instance(
        text, 
        predict_proba_multilingual, 
        labels=[0, 1, 2],
        num_features=8)

    # 0 negative, 1 neutral, 2 positive
    neg_feats = exp.as_list(label=0) if 0 in exp.local_exp else []
    neu_feats = exp.as_list(label=1) if 1 in exp.local_exp else []
    pos_feats = exp.as_list(label=2) if 2 in exp.local_exp else []

    for w, weight in neg_feats:
        class_counters["negative"][w] += abs(weight)
    for w, weight in neu_feats:
        class_counters["neutral"][w] += abs(weight)
    for w, weight in pos_feats:
        class_counters["positive"][w] += abs(weight)

    rows.append({
        "index": i,
        "text": text,
        "pred_label": pred_label,
        "neg_top": ", ".join([w for w, _ in neg_feats]),
        "neu_top": ", ".join([w for w, _ in neu_feats]),
        "pos_top": ", ".join([w for w, _ in pos_feats]),
    })

df_lime = pd.DataFrame(rows)
df_lime.to_csv("lime_explanations_first5.csv", index=False, encoding="utf-8-sig")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import re
from collections import Counter
import pandas as pd

# 直接复用当前已存在的多语言模型路径
tokenizer_multi = AutoTokenizer.from_pretrained(nlptown_bert_local_model_path)
model_multi = AutoModelForSequenceClassification.from_pretrained(
    nlptown_bert_local_model_path
)
model_multi.to(device)
model_multi.eval()


### 方法1： Integrated Gradients + transformers_interpret

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers_interpret import SequenceClassificationExplainer
from collections import Counter
import pandas as pd
import torch

# ========== 1. 加载模型和 tokenizer ==========
model_name = nlptown_bert_local_model_path
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# ========== 2. 创建 Explainer（默认使用 Layer Integrated Gradients） ==========
cls_explainer = SequenceClassificationExplainer(model, tokenizer)

# ========== 3. 与原来结构保持一致 ==========
label_map = {1: "positive", 0: "negative", -1: "neutral"} 
df_eval_preds_multi = pd.DataFrame({
    "text": X_test.tolist(),
    "true_label": y_test.tolist(),
    "pred_label": [label_map[p] for p in preds_multi]
})

max_samples = min(5, len(df_eval_preds_multi))
class_counters = {
    "negative": Counter(),
    "neutral": Counter(),
    "positive": Counter()
}
rows = []

for i in range(max_samples):
    text = df_eval_preds_multi.loc[i, "text"]
    pred_label = df_eval_preds_multi.loc[i, "pred_label"]

    # 对预测类别做归因（也可以指定 class_name="LABEL_0" 等）
    word_attributions = cls_explainer(text, class_name=None)  # None 表示用模型预测的类别

    # 过滤掉特殊 token，并按绝对贡献排序取 top
    filtered = [(w, s) for w, s in word_attributions 
                if w not in ["[CLS]", "[SEP]", "[PAD]", tokenizer.cls_token, tokenizer.sep_token]]
    top_feats = sorted(filtered, key=lambda x: abs(x[1]), reverse=True)[:8]

    # 根据预测标签累计触发词
    if pred_label in class_counters:
        for w, weight in top_feats:
            class_counters[pred_label][w] += abs(weight)

    rows.append({
        "index": i,
        "text": text,
        "pred_label": pred_label,
        "top_trigger_words": ", ".join([f"{w}({s:.3f})" for w, s in top_feats]),
        "attributions": top_feats
    })

df_ig = pd.DataFrame(rows)
df_ig.to_csv("ig_explanations_first5.csv", index=False, encoding="utf-8-sig")

# 查看全局触发词统计
print("=== Negative 触发词 ===")
print(class_counters["negative"].most_common(20))
print("\n=== Neutral 触发词 ===")
print(class_counters["neutral"].most_common(20))
print("\n=== Positive 触发词 ===")
print(class_counters["positive"].most_common(20))

=== Negative 触发词 ===
[]

=== Neutral 触发词 ===
[('ok', 0.9327455759048462), ('but', 0.5796388387680054), ('nice', 0.5142478346824646), ('.', 0.40027472376823425), ('would', 0.2811709940433502), ('made', 0.20373347401618958), ('plastic', 0.19496901333332062), ('[UNK]', 0.18629460036754608), ('really', 0.1606902927160263), ('simple', 0.13980139791965485), ('crack', 0.13895505666732788), ('were', 0.12715035676956177), ('didn', 0.11588887870311737), ('difference', 0.06455735117197037)]

=== Positive 触发词 ===
[('.', 1.7317644357681274), ('perfect', 0.8515018224716187), ('my', 0.5880635380744934), ('was', 0.42774498462677), ('reduction', 0.4063018262386322), ('happy', 0.40480557084083557), ('really', 0.3619462549686432), ('service', 0.31516000628471375), ('!', 0.2349969446659088), ('five', 0.20776960253715515), ('off', 0.20576420426368713), ('with', 0.19057750701904297), ('less', 0.18981558084487915), ('##igue', 0.14858175814151764), ('down', 0.140875443816185), ('stars', 0.1382610946893692), (

### 方法2： Integrated Gradients + Captum 

In [ ]:
from captum.attr import LayerIntegratedGradients
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import torch.nn.functional as F
from collections import Counter
import pandas as pd
import numpy as np
from string import punctuation

model_name = nlptown_bert_local_model_path
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 定义需要过滤的词
STOP_TOKENS = set([
    tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token, tokenizer.unk_token,
    "[CLS]", "[SEP]", "[PAD]", "[UNK]", "##",  # 常见特殊token
])

# 标点 + 纯数字 + 过短token
def is_valid_token(token, score=None):
    if token is None:
        return False
    token = str(token).replace("##", "").strip()
    if token in STOP_TOKENS:
        return False
    if token in punctuation or token == "":
        return False
    if token.isdigit():
        return False
    if len(token) <= 1:
        return False
    return True


# 自定义 forward：输出 [neg, neut, pos] 三个概率
def forward_func(input_ids, attention_mask):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    probs = F.softmax(logits, dim=-1)          # 假设模型是 5 类 (1~5 star)
    
    # 映射到 3 类（根据你原来的逻辑调整）
    neg = probs[:, 0] + probs[:, 1]            # 1 star + 2 star
    neut = probs[:, 2]                         # 3 star
    pos = probs[:, 3] + probs[:, 4]            # 4 star + 5 star
    return torch.stack([neg, neut, pos], dim=1)

# 对 embedding 层做 Layer IG
lig = LayerIntegratedGradients(forward_func, model.get_input_embeddings())

def get_attributions(text, target_class=None, n_steps=10):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    
    # baseline 用 [PAD]
    baseline_ids = torch.full_like(input_ids, tokenizer.pad_token_id)
    
    # 如果没指定 target，就用模型预测的最大类
    if target_class is None:
        with torch.no_grad():
            probs = forward_func(input_ids, attention_mask)
            target_class = torch.argmax(probs, dim=1).item()
    
    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        additional_forward_args=(attention_mask,),
        target=target_class,
        n_steps=n_steps,
        return_convergence_delta=True
    )
    
    # 对 embedding 维度求和，得到每个 token 的贡献
    attributions = attributions.sum(dim=-1).squeeze(0)
    attributions = attributions / torch.norm(attributions)   # 可选归一化
    
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    return list(zip(tokens, attributions.detach().cpu().numpy())), target_class

# ========== 主循环（与你原来几乎一样） ==========
class_counters = {"negative": Counter(), "neutral": Counter(), "positive": Counter()}
class_names = ["negative", "neutral", "positive"]
rows = []

max_samples = min(100, len(df_eval_preds_multi))

for i in range(max_samples):
    text = df_eval_preds_multi.loc[i, "text"]
    pred_label = df_eval_preds_multi.loc[i, "pred_label"]

    # 先拿到 3 个类别的解释结果
    class_to_attrs = {}
    for cls_idx, cls_name in enumerate(["negative", "neutral", "positive"]):
        attrs, _ = get_attributions(text, target_class=cls_idx)
        filtered = [(w, float(s)) for w, s in attrs if is_valid_token(w, s)]
        top_feats = sorted(filtered, key=lambda x: abs(x[1]), reverse=True)[:10]
        class_to_attrs[cls_name] = top_feats

    # 只保留当前预测类别对应的解释
    neg_top = ""
    neu_top = ""
    pos_top = ""

    if pred_label == "negative":
        neg_top = ", ".join([f"{w}({s:.3f})" for w, s in class_to_attrs["negative"]])
    elif pred_label == "neutral":
        neu_top = ", ".join([f"{w}({s:.3f})" for w, s in class_to_attrs["neutral"]])
    elif pred_label == "positive":
        pos_top = ", ".join([f"{w}({s:.3f})" for w, s in class_to_attrs["positive"]])

    # 也可按需要再累计到 class_counters
    if pred_label in class_counters:
        for w, weight in class_to_attrs[pred_label]:
            clean_w = str(w).replace("##", "").lower()
            class_counters[pred_label][clean_w] += abs(weight)

    rows.append({
        "index": i,
        "text": text,
        "pred_label": pred_label,
        "negative_top_trigger_words": neg_top,
        "neutral_top_trigger_words": neu_top,
        "positive_top_trigger_words": pos_top,
    })


df_ig = pd.DataFrame(rows)
df_ig.to_csv("ig_explanations_first5_v4.csv", index=False, encoding="utf-8-sig")

print(class_counters["positive"].most_common(15))

[('five', 4.888239912688732), ('great', 4.456179711967707), ('nice', 3.5102035850286484), ('and', 3.4931257243733853), ('perfect', 2.7317544519901276), ('my', 2.640038078650832), ('it', 2.560350753366947), ('love', 2.52828885614872), ('was', 2.3117991127073765), ('these', 2.158886879682541), ('good', 2.1385679077357054), ('they', 1.995553344488144), ('able', 1.8414322342723608), ('the', 1.7155152708292007), ('stars', 1.6382260546088219)]


### 汇总NLP模型的输出结果（保存和汇总上述cell的NLP输出结果）

In [ ]:
import math
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, f1_score
import pandas as pd
import numpy as np
from tqdm import tqdm

def compute_metrics(y_true, y_pred):
    y_pred = np.array(y_pred)
    y_true = np.array(y_true)
    total = len(y_pred)
    mask = (y_pred != -1)
    evaluated = mask.sum()
    neutral_pct = 0.0 if total == 0 else (1.0 - evaluated/total)
    if evaluated == 0:
        return {
            "samples_total": int(total),
            "samples_evaluated": int(evaluated),
            "neutral_pct": neutral_pct,
            "accuracy": None,
            "macro_f1": None,
            "neg_f1": None,
            "pos_f1": None,
            "neg_precision": None,
            "neg_recall": None,
            "pos_precision": None,
            "pos_recall": None,
        }
    yt = y_true[mask]
    yp = y_pred[mask].astype(int)
    acc = accuracy_score(yt, yp)
    macro = f1_score(yt, yp, average='macro', zero_division=0)
    p, r, f, s = precision_recall_fscore_support(yt, yp, labels=[0,1], zero_division=0)

    return {
        "samples_total": int(total),
        "samples_evaluated": int(evaluated),
        "neutral_pct": float(neutral_pct),
        "accuracy": float(acc),
        "macro_f1": float(macro),
        "neg_f1": float(f[0]),
        "pos_f1": float(f[1]),
        "neg_precision": float(p[0]),
        "neg_recall": float(r[0]),
        "pos_precision": float(p[1]),
        "pos_recall": float(r[1]),
    }


def normalize_preds(preds):
    out = []
    for p in preds:
        if p is None or (isinstance(p, float) and math.isnan(p)):
            out.append(-1)
        else:
            out.append(int(p))
    return out

# 假设你已经有这些变量：y_test, y_pred (tfidf), vader_pred, tb_pred, tb_nb_pred, preds_roberta, preds_multi, preds_distil
precomputed = {
    "TFIDF_LogReg": list(y_pred),               # 来自之前 pipe.predict(X_test)
    "VADER": list(vader_pred),                  # 来自 X_test.apply(vader_label)
    "TextBlob_Pattern": list(tb_pa_pred),          # 来自 X_test.apply(textblob_pattern_label)
    "TextBlob_NB": list(tb_nb_pred),            # 来自 X_test.apply(textblob_nb_label)
    "Twitter_RoBERTa": list(preds_roberta),     # 来自 evaluate_model(...) 的返回值或保存的 preds_roberta
    "Multilingual_BERT": list(preds_multi),
    "DistilBERT": list(preds_distil),
}

rows = []
for name, preds in precomputed.items():
    # 统一把 None -> -1，确保长度与 y_test 一致
    preds = normalize_preds(preds)
    metrics = compute_metrics(list(y_test), preds)   # 使用你 notebook 中已定义的 compute_metrics
    metrics.update({"model": name})
    rows.append(metrics)

summary_df = pd.DataFrame(rows)[["model","samples_total","samples_evaluated","neutral_pct","accuracy","macro_f1",
                                "neg_precision","neg_recall","neg_f1","pos_precision","pos_recall","pos_f1"]]
print(summary_df)
summary_df.to_csv("customer_review_nlp_models_output_summary.csv", index=False, encoding="utf-8-sig")

               model  samples_total  samples_evaluated  neutral_pct  accuracy  \
0       TFIDF_LogReg           7389               7389     0.000000  0.967790   
1              VADER           7389               7047     0.046285  0.892153   
2   TextBlob_Pattern           7389               7037     0.047638  0.868552   
3        TextBlob_NB           7389               7389     0.000000  0.696847   
4    Twitter_RoBERTa           7389               7051     0.045744  0.975890   
5  Multilingual_BERT           7389               7188     0.027203  0.989844   
6         DistilBERT           7389               7389     0.000000  0.932738   

   macro_f1  neg_precision  neg_recall    neg_f1  pos_precision  pos_recall  \
0  0.954911       0.891922    0.973252  0.930814       0.992134    0.966226   
1  0.805411       0.888764    0.544766  0.675491       0.892643    0.982306   
2  0.753820       0.888587    0.436874  0.585759       0.866212    0.985199   
3  0.606204       0.364711    0.487

In [64]:
import gc
gc.collect()

2250